# Notebook 02 — Systematic Exploration (Complaints)

**Purpose:** NHTSA Complaints API hypotheses verification after initial exploration.

**Hypotheses to verify:**
1. Date format MM/DD/YYYY opposite to Recalls (observed "05/21/2026" for example, will be confirmed on bigger sample)
2. Hard cap existance (for example 1767 complaints for Honda Accord 2018 based on initial exploration) (will be confirmed on bigger sample)
3. Null distribution across fields (small initial exploration sample showed none, need more records)
4. Distribution of complaints per partial VIN within a (make × model × year) — concentrated in few cohorts or evenly spread?

**Verified:**
1. `products` list structure: confirmation  multiple elements existance (`Vehicle + Tire`, `Vehicle + Child Seat`, `Vehicle + Equipment`, `2 × Vehicle`) using empirical test. Scope decision: filter to `type = "Vehicle"`. Test code retained below for reference.


**Sample plan:** ~10 - 20 (make x model x year) combinations

**Hypothesis context:** see Complaints section in  `data_profiling.md` for full context.

In [32]:
import requests
import pandas as pd
from collections import Counter
from pprint import pprint

BASE_URL = "https://api.nhtsa.gov/complaints/complaintsByVehicle"


def fetch_complaints(make, model, year):
    params = {"make": make, "model": model, "modelYear": year}
    response = requests.get(BASE_URL, params = params)
    response.raise_for_status()
    return response.json().get("results", [])

def fetch_complaint_count(make, model, year):
    params = {"make": make, "model": model, "modelYear": year}
    response = requests.get(BASE_URL, params=params)
    response.raise_for_status()
    return response.json().get("count", 0)

In [ ]:
# Already verified structure of `products` list. 
# Empirical test confirmed that `products` can be multiple-element list with `type` values as: `Vehicle`, `Tire`, `Child Seat`, `Equipment`.
# Scope decision: filter to `type = "Vehicle"` (the quality intelligence platform is focused on OEM).

products_queries = [
    ("toyota", "camry", 2020),
    ("ford", "explorer", 2022),
    ("honda", "civic", 2017),
    ("chevrolet", "malibu", 2010),
    ("nissan", "altima", 2020),
    ("hyundai", "tucson", 2023),
    ("tesla", "model 3", 2022),
    ("kia", "sportage", 2022),
    ("subaru", "outback", 2021),
    ("jeep", "grand cherokee", 2020)
]

all_complaints = []
for make, model, year in products_queries:
    complaints = fetch_complaints(make, model, year)
    all_complaints.extend(complaints)

for complaint in all_complaints:
    products = complaint.get("products", [])

    if len(products) > 1:
        types = {p.get("type") for p in products}

        print(
            f"ODI: {complaint['odiNumber']}, "
            f"number of products: {len(products)}, "
            f"types: {types}"
        )

ODI: 11647348, number of products: 2, types: {'Vehicle'}
ODI: 11636054, number of products: 2, types: {'Vehicle'}
ODI: 11614694, number of products: 2, types: {'Vehicle', 'Child Seat'}
ODI: 11490767, number of products: 2, types: {'Vehicle', 'Tire'}
ODI: 11470601, number of products: 2, types: {'Vehicle', 'Tire'}
ODI: 11671822, number of products: 2, types: {'Vehicle'}
ODI: 11637972, number of products: 2, types: {'Vehicle', 'Tire'}
ODI: 11610242, number of products: 2, types: {'Vehicle', 'Child Seat'}
ODI: 11508812, number of products: 2, types: {'Vehicle', 'Tire'}
ODI: 11711507, number of products: 2, types: {'Vehicle', 'Tire'}
ODI: 11624221, number of products: 2, types: {'Vehicle', 'Tire'}
ODI: 11562570, number of products: 2, types: {'Vehicle'}
ODI: 11537143, number of products: 2, types: {'Vehicle'}
ODI: 11518371, number of products: 2, types: {'Vehicle'}
ODI: 11482127, number of products: 2, types: {'Vehicle', 'Equipment'}
ODI: 11317427, number of products: 2, types: {'Vehicle',

In [ ]:
# Hypothesis 1 test: US date format present in `dateOfIncident` and `dateComplaintFiled` (for example, "05/31/2026")
# Date format of `Complaints' is different than in `Recalls` where date is in EU format.

all_records = []
for make, model, year in products_queries:
    complaints = fetch_complaints(make, model, year)
    all_records.extend(complaints)

print(f"Total records collected: {len(all_records)}")
print(f"Total  (make x model x year) combinations examined: {len(products_queries)}")

Total records collected: 4021
Total  (make x model x year) combinations examined: 10


In [ ]:
# Parse dates and check format consistency
date_analysis = []

for record in all_records:
    complaint = record.get("odiNumber", "")
    incident_date_str = record.get("dateOfIncident", "")
    filed_date_str = record.get("dateComplaintFiled", "")

    
    if not incident_date_str or "/" not in incident_date_str:
        continue  # skip if no date

    if not filed_date_str or "/" not in filed_date_str:
        continue  # skip if no date
        
    incident_parts = incident_date_str.split("/")
    if len(incident_parts) != 3:
        continue  # skip if not formatted correctly
        
    incident_first, incident_middle, incident_last = incident_parts

    filed_parts = filed_date_str.split("/")
    if len(filed_parts) != 3:
        continue  # skip if not formatted correctly
        
    filed_first, filed_middle, filed_last = filed_parts
        
        
    date_analysis.append({
        "complaint": complaint,
        "incident_date_raw": incident_date_str,
        "incident_first": int(incident_first),
        "incident_middle": int(incident_middle),
        "filed_date_raw": filed_date_str,
        "filed_first": int(filed_first),
        "filed_middle": int(filed_middle)
    })

# Convert to DataFrame for analysis
df_dates = pd.DataFrame(date_analysis)
print(f"Parsed records: {len(df_dates)}")
print()
print("Incident first component range:", df_dates["incident_first"].min(), "to", df_dates["incident_first"].max())
print("Incident middle component range:", df_dates["incident_middle"].min(), "to", df_dates["incident_middle"].max())
print("Filing first component range:", df_dates["filed_first"].min(), "to", df_dates["filed_first"].max())
print("Filing middle component range:", df_dates["filed_middle"].min(), "to", df_dates["filed_middle"].max())

Parsed records: 4021

Incident first component range: 1 to 12
Incident middle component range: 1 to 31
Filing first component range: 1 to 12
Filing middle component range: 1 to 31


In [ ]:
df_dates["incident_date"] = pd.to_datetime(df_dates["incident_date_raw"], format = '%m/%d/%Y')
df_dates["filed_date"] = pd.to_datetime(df_dates["filed_date_raw"], format = '%m/%d/%Y')

dates_mismatch = len(df_dates[df_dates["incident_date"] > df_dates["filed_date"]])

df_dates['filing_lag_days'] = (df_dates["filed_date"] - df_dates["incident_date"]).dt.days

print(f"Number of date mismatch (an issue filed before an incident): {dates_mismatch}")
print(f"Average filing lag [days]: {df_dates['filing_lag_days'].mean():.2f}")
df_dates['filing_lag_days'].describe()

Number of date mismatch (an issue filed before an incident): 0
Average filing lag [days]: 123.93


count     4021.000000
mean       123.927381
std        780.129226
min          0.000000
25%          1.000000
50%          8.000000
75%         53.000000
max      20562.000000
Name: filing_lag_days, dtype: float64

### Hypothesis 1 — Verification Result

**Status:** Confirmed

**Findings:**
- Confirmed: `dateOfIncident` and `dateComplaintFiled` follow American date format MM/DD/YYYY based on the observation of 4021-element sample for 10 (make x model x year) combinations. First component of the date represents the values from the range **1 to 12**, and the middle one **1 to 31** indicating respectively month and date. 
Date mismatch, understood as an issue filed before an incident, is not present in the larger dataset.
Filing lag shows highly skewed distribution: median 8 days indicates most complaints are filed within a week of incident, but mean of about 124 days reveals a long tail of retroactive filings (likely complaints filed in response to publicized recalls or after problem recurrence). This skew has implications: median is the more representative central tendency for this metric.

**Architectural implication:** 
- Silver layer: parse `dateOfIncident` and `dateComplaintFiled` using `pd.to_datetime(..., format="%m/%d/%Y")`.
Cross-source JOIN with `Recalls` (parsed separately with DD/MM/YYYY format) operates on standardized DATE types, not on raw strings. No string-format conversion needed — both endpoints are normalized to the same internal representation
- Gold layer: `fact_complaint` will link to `dim_date` via surrogate integer key (`incident_date_key`, `filed_date_key` in YYYYMMDD format) 
- Silver layer derived field: `filing_lag_days = filed_date - incident_date` 
- Potential business question enabled by cross-source JOIN: *"Which OEMs have the shortest median filing lag — indicating responsive customer communities and rapid issue reporting?"*

In [ ]:
# Hypothesis 2 test: Hard cap on records per query

max_count = 0
max_vehicle = None

edge_case_queries = [
("honda", "accord", 2008),
("honda", "accord", 2003),
("honda", "civic", 2008),
("toyota", "camry", 2008),
("toyota", "camry", 2007),
("toyota", "corolla", 2009),
("ram", "1500", 2014),
("ford", "focus", 2018),
("ford", "explorer", 2018),
("bmw", "m3", 2001)
]

for make, model, year in edge_case_queries:
    count = fetch_complaint_count(make, model, year)

    if count > max_count:
        max_count = count
        max_vehicle = (make, model, year)

print(max_count)
print(max_vehicle)

3613
('toyota', 'camry', 2007)


### Hypothesis 2 — Verification Result

**Status:** Confirmed 

**Findings:**

- Empirically tested complaints 10 (make x model x year) combinations including expected ehigh-volume (high number complaints due to well-known air bag issue) 
- Maximal complaints number observed: **3613 records** (Toyota Camry 2007)
- No round-number (e.g. 20, 50, 100) response sizes that would indicate a cap

**Architectural implication:**

- `complaintsByVehicle` endpoint at (make, model, year) granularity returns complete data in a single request. Estimated maximum response size: 3613 records × ~2-3 KB each ≈ 10 MB per request — comfortably within memory and network constraints.
If future ingestion encounters significantly larger response volumes (10x+ current observed maximum), consider partitioning queries by additional dimensions (for example, component category, time window) to maintain manageable response sizes.

In [56]:
# Hypothesis 3 test: missing values presence
# Sample: based on 10 (make × model × year) combinations

# Convert all_records list to DataFrame 
# Placehoders considered as missing values - they are also counted
df_records = pd.DataFrame(all_records)
df_records.replace(["", "N/A", "Unknown", "-", "null"], None, inplace = True)

# Missing values summary table: count and percentage sorted descending
missing_complaints_summary = pd.DataFrame({
    "missing_count": df_records.isna().sum(),
    "missing_pct": df_records.isna().mean() * 100
}).sort_values("missing_pct", ascending=False)

print(f"Total complaints records: {len(df_records)}")
print(f"Total fields: {len(df_records.columns)}\n")
missing_complaints_summary

Total complaints records: 4021
Total fields: 12



,missing_count,missing_pct
vin,175,4.352151
summary,1,0.024869
odiNumber,0,0.000000
manufacturer,0,0.000000
fire,0,0.000000
crash,0,0.000000
numberOfInjuries,0,0.000000
numberOfDeaths,0,0.000000
dateComplaintFiled,0,0.000000
dateOfIncident,0,0.000000


In [57]:
# Flatten `products` to receive an access to the rest of fields.
# Placehoders considered as missing values - they are also counted
df_exploded = df_records.explode("products").reset_index(drop = True)
df_products = pd.json_normalize(df_exploded["products"])
df_products.replace(["", "N/A", "Unknown", "-", "null"], None, inplace = True)

missing_products_summary = pd.DataFrame({
    "missing_count": df_products.isna().sum(),
    "missing_pct": df_products.isna().mean() * 100
}).sort_values("missing_pct", ascending=False)

print(f"Total products records: {len(df_products)}")
print(f"Total fields: {len(df_products.columns)}\n")
missing_products_summary

Total products records: 4050
Total fields: 6



,missing_count,missing_pct
size,4042,99.802469
type,0,0.000000
productYear,0,0.000000
productMake,0,0.000000
productModel,0,0.000000
manufacturer,0,0.000000


In [58]:
df_products_only = pd.json_normalize(df_exploded["products"])
print(df_products_only.groupby("type")["size"].apply(lambda s: s.notna().sum()))

type
Child Seat    0
Equipment     0
Tire          8
Vehicle       0
Name: size, dtype: int64


In [60]:
for ptype in df_products_only["type"].unique():
    subset = df_products_only[df_products_only["type"] == ptype]
    non_null_cols = [c for c in subset.columns if subset[c].notna().any()]
    print(f"{ptype}: {non_null_cols}")

Vehicle: ['type', 'productYear', 'productMake', 'productModel', 'manufacturer']
Child Seat: ['type', 'productYear', 'productMake', 'productModel', 'manufacturer']
Tire: ['type', 'productYear', 'productMake', 'productModel', 'manufacturer', 'size']
Equipment: ['type', 'productYear', 'productMake', 'productModel', 'manufacturer']


### Hypothesis 3 — Verification Result

**Status:** Confirmed

**Findings:**
- Confirmed: Missing values are present in the sample of 10 (make x model x year) combination in 4021 complaints.
`vin` is missing in ~4.35% (174 cases per 4021) of records - VIN can be optional in the NHTSA complaint form.
`summary` is missing in ~0.02% (1 case per 4021) which is irrelevant statistically.
- The `size` field, present in flattened products data, is missing in ~99.80% of rows. Verified to be **type-specific** to `Tire` products only — all 8 Tire products in the sample have a `size` value - Vehicle, Child Seat, and Equipment products do not use this field. With the Vehicle-only scope filter, `size` becomes irrelevant to the project.

**Architectural implication:** 
- **Hard constraints** in silver layer: `odiNumber`, `manufacturer`, both dates, all severity fields, `components` — enforce as `not_null` tests
- **Soft constraint**: `vin` — nullable; analyses depending on VIN (plant code, cohort concentration) implicitly filter to records with VIN


In [111]:
# Hypothesis 4 test: concentration of distribution of complaints per partial VIN within a (make × model × year)

def extract_vehicle(products_list):
    for p in products_list:
        if p.get("type") == "Vehicle":
            return p
    return {}  # fallback if no Vehicle product

df_records["vehicle"] = df_records["products"].apply(extract_vehicle)
df_records["product_make"] = df_records["vehicle"].apply(lambda v: v.get("productMake"))
df_records["product_model"] = df_records["vehicle"].apply(lambda v: v.get("productModel"))
df_records["product_year"] = df_records["vehicle"].apply(lambda v: v.get("productYear"))


# Filter to valid VINs only and apply uppercase normalization
df_valid_vin = df_records[df_records["vin"].notna()].copy()
df_valid_vin["vin"] = df_valid_vin["vin"].str.upper()

# Length check
df_valid_vin = df_valid_vin[df_valid_vin["vin"].str.len() == 11]

# Alphanumeric check
df_valid_vin = df_valid_vin[df_valid_vin["vin"].str.isalnum()]

# True cohort extraction
df_valid_vin ["true_cohort"] = df_records["vin"].str[:3] + df_records["vin"].str[9:11]

print(f"Total complaints: {len(df_records)}")
print(f"With VIN present: {df_records['vin'].notna().sum()}")
print(f"With valid VIN: {len(df_valid_vin)}")
print(f"Invalid VIN rate: {(df_records['vin'].notna().sum() - len(df_valid_vin)) / df_records['vin'].notna().sum() * 100:.2f}%")

Total complaints: 4021
With VIN present: 3846
With valid VIN: 3839
Invalid VIN rate: 0.18%


In [112]:
# How much complaints has one partial VIN?
labeled_cohort = (
    df_valid_vin.groupby("true_cohort").agg(
        n_records=("vin", "size"),
        product_make=("product_make", "first"),
        product_model=("product_model", "first"),
        product_year=("product_year", "first")
    ).reset_index().sort_values("n_records", ascending = False)
)
labeled_cohort.head(10)

,true_cohort,n_records,product_make,product_model,product_year
21,1G1AF,909,CHEVROLET,MALIBU,2010
51,5YJNF,664,TESLA,MODEL 3,2022
20,1G1A4,270,CHEVROLET,MALIBU,2010
3,19XHE,256,HONDA,CIVIC,2017
42,4S4M3,248,SUBARU,OUTBACK,2021
44,4T1LU,241,TOYOTA,CAMRY,2020
12,1FMNG,221,FORD,EXPLORER,2022
6,1C4LC,201,JEEP,GRAND CHEROKEE,2020
38,2HGHH,168,HONDA,CIVIC,2017
26,1N4LC,135,NISSAN,ALTIMA,2020


In [113]:
labeled_cohort.describe()

,n_records
count,68.000000
mean,56.455882
std,148.298553
min,1.000000
25%,1.000000
50%,2.000000
75%,17.500000
max,909.000000


### Hypothesis 4 — Verification Result

**Status:** Confirmed

**Findings:**

Production cohort concentration confirmed on a 3839-record validated sample (after VIN length check, alphanumeric validation, and uppercase normalization):
- 68 unique production cohorts identified `WMI + model year + plant code` as `true_cohort` (positions 1-3, 10, 11 of partial VIN).
- Distribution highly skewed: median 2 complaints per cohort, but top cohorts reach more than 900 complaints (which increases the mean value to ~56)
- Top 3 cohorts (Tesla Model 3 2022 - `F`: Fremont, California, Chevrolet Malibu 2010 - `F`: Fairfax Assembly, Kansas City, Kansas and `4`: Orion Assembly, Lake Orion, Michigan) account for ~48% of all complaints in sample.

**Domain interpretation:**

- **Cross-manufacturer comparison**: Tesla Model 3 has shorter field exposure than Chevrolet Malibu. Higher complaint density for Tesla may be influenced by multiple factors, including vehicle characteristics, consumer reporting behavior, and software-related issues associated with OTA-update vehicles.
- **Cross-plant comparison**: Chevrolet Malibu 2010 was produced at two plants with 3.4× difference in complaint volume — exactly the type of plant-level quality differential that automotive process audits aim to identify.

**Architectural implications:**

- Gold layer: `dim_production_cohort` table with (`WMI + year + plant key`), linked to fact_complaint
- Derived metric: complaints-per-cohort distribution by manufacturer — flags plants/years with anomalous patterns
- Process audit alignment — answers questions like:
 *Which production locations / model years exhibit elevated field issues?* 
 *Where exists opportunity for cross-plant quality improvement?*
 *Should benchmarking sessions be initiated between plants of the same model?*